# Generators sharing a bus in the MATPOWER reader

A MATPOWER case may put several generators on one bus, may label a
generator's bus PQ, and may carry generators that are switched off. This
notebook builds such a case from the public case9 reference and checks that
the reader maps every online generator to its own component.


## Fetching a reference case


In [ ]:
import urllib.request

import numpy as np
import scipy.io

import dpsim

base_url = "https://raw.githubusercontent.com/dpsim-simulator/reference-results/master/Matpower/"
urllib.request.urlretrieve(base_url + "case9results.mat", "case9results.mat")

mpc = scipy.io.loadmat("case9results.mat", simplify_cells=True)["case9results"]
gen = np.atleast_2d(np.array(mpc["gen"], dtype=float))
bus = np.atleast_2d(np.array(mpc["bus"], dtype=float))
print("buses", bus.shape[0], "generators", gen.shape[0])

## Adding more units to the case

The MATPOWER generator columns used below are bus, Pg, Qg, Qmax, Qmin, Vg,
mBase and status. Bus 2 gets three units instead of one, bus 5 is labelled PQ
but is given two units, and one unit at bus 2 is switched off. The second unit
at bus 5 is left with an mBase of 0, which MATPOWER reads as the system base.


In [ ]:
units = [
    # bus,   Pg,   Qg, Qmax,  Qmin,     Vg, mBase, status
    (2.0, 60.0, 5.0, 40.0, -10.0, 1.025, 100.0, 1.0),
    (2.0, 55.0, 7.0, 35.0, -12.0, 1.025, 150.0, 1.0),
    (2.0, 48.0, 9.0, 30.0, -14.0, 1.025, 200.0, 1.0),
    (5.0, 12.0, 3.0, 20.0, -20.0, 1.000, 0.0, 1.0),
    (5.0, 8.0, 2.0, 15.0, -15.0, 1.000, 250.0, 1.0),
    (2.0, 999.0, 99.0, 99.0, -99.0, 1.025, 100.0, 0.0),
]

rows = [row for row in gen if int(row[0]) != 2]
for unit in units:
    row = gen[0].copy()
    row[:8] = unit
    rows.append(row)

mpc["gen"] = np.vstack(rows)
scipy.io.savemat("multigen.mat", {"multigen": mpc})
print("generator rows now", mpc["gen"].shape[0])

## Every online generator becomes its own component

The first generator at a bus keeps the unsuffixed name it always had, so cases
with one generator per bus are unaffected. Further units are suffixed.


In [ ]:
reader = dpsim.matpower.Reader("multigen.mat", "multigen")
system = reader.load_mpc()

generators = sorted(c.name() for c in system.components if c.name().startswith("Gen_N"))
print(generators)

assert generators == [
    "Gen_N2",
    "Gen_N2_1",
    "Gen_N2_2",
    "Gen_N3",
    "Gen_N5",
    "Gen_N5_1",
]

## Each component carries its own row

A generator must not inherit the set points of the first machine at its bus.
The unit with an mBase of 0 falls back to the system base power.


In [ ]:
mw_w = 1e6
expected = {
    "Gen_N2": (60.0, 5.0, 40.0, -10.0),
    "Gen_N2_1": (55.0, 7.0, 35.0, -12.0),
    "Gen_N2_2": (48.0, 9.0, 30.0, -14.0),
    "Gen_N5": (12.0, 3.0, 20.0, -20.0),
    "Gen_N5_1": (8.0, 2.0, 15.0, -15.0),
}

for name, (p, q, q_max, q_min) in expected.items():
    component = system.component(name)
    assert np.isclose(component.attr("P_set").get(), p * mw_w)
    assert np.isclose(component.attr("Q_set").get(), q * mw_w)
    assert np.isclose(component.attr("Q_max").get(), q_max * mw_w)
    assert np.isclose(component.attr("Q_min").get(), q_min * mw_w)

print("every generator kept its own set points")

## Offline generators are dropped

The switched-off unit at bus 2 would inject 999 MW if it were mapped, so it
must not appear as a component and must not reach the reference table.


In [ ]:
assert "Gen_N2_3" not in generators
assert (reader.mpc_gen_data["status"] == 1).all()
print("offline generators are absent")

## Generators on a PQ-labelled bus

Bus 5 is labelled PQ but carries generation. Mapping it is the default and can
be turned off to recover the previous behaviour.


In [ ]:
reader_pq_off = dpsim.matpower.Reader("multigen.mat", "multigen")
system_pq_off = reader_pq_off.load_mpc(map_pq_bus_generators=False)

without_pq = sorted(
    c.name() for c in system_pq_off.components if c.name().startswith("Gen_N")
)
print(without_pq)

assert without_pq == ["Gen_N2", "Gen_N2_1", "Gen_N2_2", "Gen_N3"]

## The reference table sums co-located generation

Bus 2 carries three units, so its reported generation is their sum rather than
the first unit's contribution alone.


In [ ]:
results = reader.get_pf_results()
print(results.to_string())

bus2 = results.loc[results["Bus"] == "N2"].iloc[0]
assert np.isclose(bus2["P [MW]"], 60.0 + 55.0 + 48.0)

## Initialising from the power flow results

Each generator row must find the component it was mapped to, otherwise units
sharing a bus overwrite one another.


In [ ]:
counter = {}
names = []
for index, _ in reader.mpc_gen_data.iterrows():
    bus_number = reader.mpc_gen_data.at[index, "bus"]
    gen_i = counter.get(bus_number, 0)
    counter[bus_number] = gen_i + 1
    names.append(reader.gen_component_name(bus_number, gen_i))

assert len(set(names)) == len(names)
assert sorted(n for n in names if system.component(n) is not None) == generators

reader.init_from_pf_results()
print("initialised", len(generators), "generators from", len(names), "rows")